# Resume comparison: "Who is most like me"

![0Resumes](0Resumes.png)

## Get resumes in consumable format 
- ( see build_resume_corpus.ipynb )

## Corpus similarity (embeddings)

#### Step 1:  
- Loads [`output/corpus_20_resumes.csv`](output/corpus_20_resumes.csv), encodes **each row** in **Summary**, **Experience**, and **Skills** with `all-MiniLM-L6-v2`, adds an **`embedding`** column (JSON array per row), and writes **`output/corpus_with_embedding.csv`**.

- Set **`TARGET_RESUME_ID`** below to choose which resume is “me” for the optional ranking cell.

In [2]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd().resolve()
if (_cwd / "corpus_similarity.ipynb").exists():
    NOTEBOOK_DIR = _cwd
elif (_cwd / "peopleLikeMe" / "corpus_similarity.ipynb").exists():
    NOTEBOOK_DIR = _cwd / "peopleLikeMe"
else:
    NOTEBOOK_DIR = _cwd

OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = OUTPUT_DIR / "corpus_20_resumes.csv"
META_PATH = OUTPUT_DIR / "resume_metadata.csv"
OUT_EMB_PATH = OUTPUT_DIR / "corpus_with_embedding.csv"

TARGET_RESUME_ID = 1  # change to any ResumeID in the corpus

df = pd.read_csv(CORPUS_PATH, dtype={"ResumeID": int, "ItemID": int})
df["Title"] = df["Title"].fillna("").astype(str)

CORPUS_PATH.resolve(), len(df), sorted(df["ResumeID"].unique())[:5], "...", df["Section"].unique().tolist()

(WindowsPath('C:/Users/DeloresMincarelli/Documents/Portfolio/peopleLikeMe/output/corpus_20_resumes.csv'),
 236,
 [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)],
 '...',
 ['Summary', 'Experience', 'Education', 'Skills'])

### Step 2: Install Py sentence-transformers & Scikit-learn & Embedding model (it's free)

Sentence-transformers
- A library for turning sentences or short texts into fixed-size vectors (embeddings) so you can compare meaning, not just words.
- Built on PyTorch and Hugging Face Transformers.
Pre-trained models (e.g. MiniLM, MPNet) map text to dense vectors; similar meanings tend to sit close in vector space.
Common uses: semantic search, duplicate detection, clustering, re-ranking, similarity between documents or queries.
It’s the usual choice when you want “what does this sentence mean?” style similarity without training a big model from scratch.

Scikit-learn

- The main Python library for classical / tabular machine learning and scientific computing around models.

Together: sentence-transformers gives you text embeddings; scikit-learn can use those vectors (or other features) for clustering, classification, nearest neighbors, etc.

Load embedding model:  all-MiniLM-L6-v2 (from SentenceTransformers) because it’s:

- Built specifically for semantic similarity
It converts text → vectors (embeddings)
Similar meaning → vectors close together (cosine similarity)
That’s exactly what your workflow needs:
Resume bullets ↔ skills/tasks matching
- Very efficient
Small model (~80MB)
Fast on CPU (no GPU needed)
Great for batching thousands of rows (like O*NET)
- “Good enough” accuracy
- Is it free?
Yes — completely free
Open-source (Apache 2.0 license)
No API calls
Runs locally
No usage limits
No cost per embedding
That’s why it’s commonly used in: prototypes and offline pipelines

In [2]:
# pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


### Step 3:  Encode Summary, Experience, and Skills (one embedding per row)

Education rows are kept in the export with an **empty** `embedding`. Vectors are stored as **JSON arrays** in the CSV.

In [3]:
ENCODE_SECTIONS = {"Summary", "Experience", "Skills"}


def row_to_encode_text(row: pd.Series) -> str:
    section = str(row["Section"]).strip()
    text = str(row["Text"]).strip()
    title = str(row["Title"]).strip() if pd.notna(row["Title"]) else ""
    if section == "Summary":
        return text
    if section == "Experience":
        return f"{title}: {text}" if title else text
    if section == "Skills":
        return text
    return text


mask = df["Section"].isin(ENCODE_SECTIONS)
encode_idx = df.index[mask]
texts = [row_to_encode_text(df.loc[i]) for i in encode_idx]

vectors = model.encode(texts, show_progress_bar=True)

df_out = df.copy()
df_out["embedding"] = ""
for row_i, idx in enumerate(encode_idx):
    df_out.at[idx, "embedding"] = json.dumps(vectors[row_i].tolist())

df_out.to_csv(OUT_EMB_PATH, index=False, encoding="utf-8", na_rep="")

encoded_rows = int(mask.sum())
print("Wrote", OUT_EMB_PATH.resolve())
print("Encoded rows:", encoded_rows, "vector dim:", vectors.shape[1] if len(vectors) else 0)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Wrote C:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\output\corpus_with_embedding.csv
Encoded rows: 196 vector dim: 384


### From ChatGPT: How to compare?

Core idea:
A resume is a structured portfolio of meanings (summary + many bullets), not one document and not interchangeable snippets. “Professionals like me” depends on what you mean by similar (overall identity vs day-to-day work vs strongest specialties vs breadth), and no single number captures all of that.

What to avoid

- Treating “embed the embeddings” as a magic second step: if you collapse many vectors to one, that should be an explicit aggregation (mean, weighted mean, role/theme structure), not an opaque second embedding model.

- Do not rely on one max bullet match or one plain average over all bullets as the whole story: it's fragile, length-sensitive, and easy to critique.

- Letting summary similarity dominate: often generic and polished.

... Instead ...

- Multi-level comparison: 
(1) summary (broad positioning), (2) weighted bullet centroid (holistic work signal), and (3) bullet-level alignment (concrete overlap), then a (4) weighted composite (they suggested something like 20% / 30% / 50%).
- For bullets: bidirectional aggregation (A→B and B→A) so broad vs narrow profiles are not confused.
Use top‑k (or proportional k) over per-bullet best matches so a few strong lines and noise from long resumes do not dominate a naive full average.
- For scrutiny: add explainability (component scores + a few supported bullet pairs)
- The defensible story is: similarity is estimated at several levels, aggregated with clear rules, and reported with evidence—not “one cosine score proves sameness.” That matches how embeddings are usually used in high-stakes settings: transparent signals, explicit limits on what the score means.



### Step 4: Compare Summaries (broad positioning)

**SummarySim** between two resumes is the **cosine similarity** of their **Summary** row embeddings (already stored in `output/corpus_with_embedding.csv`). This cell reads that CSV only (no re-encode), uses `sklearn.metrics.pairwise.cosine_similarity`, ranks every resume against `TARGET_RESUME_ID`, and merges `resume_metadata.csv` for context - this has a high level description of the resume (ex: finance, senior level)

Summary text is often polished and uses generic phrases (*results-driven*, *cross-functional*, *data-driven*), so cosine similarity here can **overstate** how alike two people are in real work. Treat this as **one component** of a future composite score, not a full “professionals like me” verdict.

**Checks:** similarity of the target to itself should be **1.0** (within float noise). All similarities lie in **[-1, 1]**.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

if not OUT_EMB_PATH.is_file():
    raise FileNotFoundError(f"Embedded corpus not found: {OUT_EMB_PATH.resolve()}")
if not META_PATH.is_file():
    raise FileNotFoundError(f"Resume metadata not found: {META_PATH.resolve()}")

emb_df = pd.read_csv(OUT_EMB_PATH, dtype={"ResumeID": int, "ItemID": int})
sum_mask = emb_df["Section"].astype(str).str.strip() == "Summary"
sum_df = emb_df.loc[sum_mask & emb_df["embedding"].astype(str).str.len().gt(0)].copy()

counts = sum_df.groupby("ResumeID").size()
all_ids = sorted(emb_df["ResumeID"].unique())
missing_summary = [rid for rid in all_ids if rid not in counts.index or counts.loc[rid] == 0]
if missing_summary:
    raise ValueError(f"No non-empty Summary embedding for ResumeID(s): {missing_summary}")
multi = counts[counts > 1]
if len(multi):
    raise ValueError(
        "Expected exactly one Summary row per ResumeID; got duplicates: "
        f"{multi.to_dict()}"
    )

ids = sorted(sum_df["ResumeID"].unique())
if TARGET_RESUME_ID not in ids:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} not in corpus summary rows")

def summary_vec(rid: int) -> np.ndarray:
    row = sum_df.loc[sum_df["ResumeID"] == rid, "embedding"].iloc[0]
    return np.array(json.loads(row), dtype=np.float32)

vectors = np.stack([summary_vec(rid) for rid in ids], axis=0)
target_row = ids.index(TARGET_RESUME_ID)
target_vec = vectors[target_row].reshape(1, -1)

sims = cosine_similarity(target_vec, vectors)[0]
rank_df = (
    pd.DataFrame({"ResumeID": ids, "summary_cosine_similarity": sims.astype(float)})
    .sort_values("summary_cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
meta = pd.read_csv(META_PATH)
rank_df = rank_df.merge(meta, on="ResumeID", how="left")

self_sim = float(sims[target_row])
assert abs(self_sim - 1.0) < 1e-5, f"expected target self-similarity ~1, got {self_sim}"
assert sims.min() >= -1.0001 and sims.max() <= 1.0001

rank_df

,ResumeID,summary_cosine_similarity,primary_domain,role_family,seniority_band
0,1,1.000000,data_analytics,healthcare_analytics,mid
1,4,0.515133,data_analytics,healthcare_reporting,early
2,7,0.499845,data_science,research_ds,senior
3,8,0.444134,data_science,nlp_search,mid
4,2,0.406697,data_analytics,bi_reporting,mid
5,9,0.361967,data_science,forecasting,mid
6,14,0.335894,finance,fpna,senior
7,3,0.330743,data_analytics,operations_analytics,senior
8,13,0.329266,hr,total_rewards,mid
9,17,0.265207,engineering,structural,mid


### Step 5: Weighted experience centroid (holistic work signal)
- What if Resume A has 10 bullets, B has 3 — what’s the impact?
Per resume: A’s centroid uses only A’s 10 vectors; B’s uses only B’s 3. Bullet counts do not get mixed when building each centroid.
Still comparable: Both centroids live in the same embedding space, so cosine between C_A and C_B is well-defined.

- How count matters (indirectly):

More bullets → the centroid is an average over more directions. Diverse bullets can pull the average toward something broader / more “middle” (sometimes described as washing out sharp specialties). Many bullets saying similar things pull the centroid strongly toward that theme (especially if weights are similar).
Fewer bullets → the centroid is dominated by those few directions; it can look more specific or noisier as a summary of the whole career, depending on what those three bullets say.
So: 10 vs 3 does not change the mathematical type (always one 384‑dim vector per resume). It changes how that vector is formed and therefore where it sits in space—and that in turn affects the cosine score when you compare A to B, but there is no separate “penalty” for 10 vs 3 in the centroid formula itself; the effect shows up through content and diversity of those bullets (and your recency weights).

**CentroidSim** compares each resume’s **single centroid vector** **C_R**: a **weighted mean** of **Experience** row embeddings from `corpus_with_embedding.csv` (same dimension as one bullet, e.g. 384). **CentroidSim** vs `TARGET_RESUME_ID` is **cosine similarity** between **C_target** and each **C_R** via `sklearn.metrics.pairwise.cosine_similarity`.

**Configurable (top of the next code cell):** three weights `WEIGHT_EXPERIENCE_RECENT` / `MID` / `OLD`, and two month cutoffs `RECENCY_MONTHS_AGO_MAX_RECENT` and `RECENCY_MONTHS_AGO_MAX_MID` from `RecencyMonthsAgo` (**NaN** → recent tier). Invalid thresholds (`MAX_RECENT >= MAX_MID`) raise **`ValueError`**.

A centroid captures **broad semantic overlap** across bullets, but **averaging can wash out** sharp specialties; **many similar bullets** pull **C_R** toward that theme. Use this as **component 2** of a future composite, not a full similarity verdict.

**Checks:** target self-similarity **≈ 1**; all cosines in **[-1, 1]**. Spot-read Experience text for a few pairs against the ranking.

### Example: What’s actually happening

Original vector:  
v = [1, 2, 3]  
Scaled vector because this bullet is recent (configurable):  
1.25 × v = [1.25, 2.5, 3.75]  
Key geometric insight  
The direction does NOT change  
Only the length (magnitude) increases  
The scaled vector lies on the same line from the origin  
Now translate that to embeddings (this is the important part)  

Think of an embedding like:

A direction in space = the meaning
The length = how strongly that meaning is expressed (or weighted)

So… does scaling change meaning?

Short answer:
👉 No — scaling does NOT change the semantic meaning

Why:

Most embedding comparisons use cosine similarity  
Cosine similarity depends only on angle, not magnitude

So:  

v and 1.25·v  
have identical cosine similarity (1.0)  
→ interpreted as exact same meaning  
Why scaling is still used (important for your centroid work)  

Scaling is not about changing meaning — it's about influence.

From your example:  

Recent bullet → weight = 1.25  
Older bullet → weight = 0.75  

When you scale:  

You’re saying:
👉 “This vector should pull the centroid more”

![vector scaling](vectorscale.png)

### Example
(your real vectors have length 384, but the steps are the same).  

Setup — Resume A  

Recent  
v₁ = [1, 2, 3]  
α₁ = 1.25  

Old  
v₂ = [1, 5, 7]  
α₂ = 0.75  

Step 1 — Scale each vector by its weight  
α₁ v₁ = 1.25 × [1, 2, 3] = [1.25, 2.5, 3.75]

α₂ v₂ = 0.75 × [1, 5, 7] = [0.75, 3.75, 5.25]  

Step 2 — Add the weighted vectors (component‑wise)  
α₁ v₁ + α₂ v₂ = [1.25 + 0.75, 2.5 + 3.75, 3.75 + 5.25] = [2, 6.25, 9]  

Step 3 — Add the weights  
α₁ + α₂ = 1.25 + 0.75 = 2  
  
Step 4 — Divide to get the centroid (weighted average)  
C_A = (α₁ v₁ + α₂ v₂) / (α₁ + α₂) = [2, 6.25, 9] / 2 = [1, 3.125, 4.5]  
  
So Resume A’s centroid is the 3‑vector C_A = [1, 3.125, 4.5].  
That is not a single number; it is one point in the same 3‑D space as the bullets.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# --- tune experience row weights (RecencyMonthsAgo → α) ---
WEIGHT_EXPERIENCE_RECENT = 1.25
WEIGHT_EXPERIENCE_MID = 1.0
WEIGHT_EXPERIENCE_OLD = 0.75
RECENCY_MONTHS_AGO_MAX_RECENT = 12
RECENCY_MONTHS_AGO_MAX_MID = 48


def recency_weight(
    months_ago: float,
    w_recent: float,
    w_mid: float,
    w_old: float,
    max_recent: float,
    max_mid: float,
) -> float:
    if max_recent >= max_mid:
        raise ValueError(
            f"RECENCY_MONTHS_AGO_MAX_RECENT ({max_recent}) must be < "
            f"RECENCY_MONTHS_AGO_MAX_MID ({max_mid})"
        )
    if pd.isna(months_ago) or float(months_ago) <= max_recent:
        return w_recent
    if float(months_ago) <= max_mid:
        return w_mid
    return w_old


if not OUT_EMB_PATH.is_file():
    raise FileNotFoundError(f"Embedded corpus not found: {OUT_EMB_PATH.resolve()}")
if not META_PATH.is_file():
    raise FileNotFoundError(f"Resume metadata not found: {META_PATH.resolve()}")

emb_df2 = pd.read_csv(OUT_EMB_PATH, dtype={"ResumeID": int, "ItemID": int})
exp_mask = emb_df2["Section"].astype(str).str.strip() == "Experience"
exp_df = emb_df2.loc[exp_mask & emb_df2["embedding"].astype(str).str.len().gt(0)].copy()

all_resume_ids = sorted(emb_df2["ResumeID"].unique())
exp_counts = exp_df.groupby("ResumeID").size()
missing_exp = [rid for rid in all_resume_ids if rid not in exp_counts.index or exp_counts.loc[rid] == 0]
if missing_exp:
    raise ValueError(f"No non-empty Experience embedding for ResumeID(s): {missing_exp}")

if exp_df["RecencyMonthsAgo"].isna().mean() > 0.5:
    print("Warning: over half of Experience rows have NaN RecencyMonthsAgo; those rows use the RECENT weight tier.")

centroids: dict[int, np.ndarray] = {}
for rid in all_resume_ids:
    rows = exp_df.loc[exp_df["ResumeID"] == rid]
    w_list = []
    v_list = []
    for _, row in rows.iterrows():
        alpha = recency_weight(
            row["RecencyMonthsAgo"],
            WEIGHT_EXPERIENCE_RECENT,
            WEIGHT_EXPERIENCE_MID,
            WEIGHT_EXPERIENCE_OLD,
            RECENCY_MONTHS_AGO_MAX_RECENT,
            RECENCY_MONTHS_AGO_MAX_MID,
        )
        v = np.array(json.loads(row["embedding"]), dtype=np.float32)
        w_list.append(alpha)
        v_list.append(v * alpha)
    w_sum = float(np.sum(w_list))
    centroids[rid] = np.sum(np.stack(v_list, axis=0), axis=0) / w_sum

if TARGET_RESUME_ID not in centroids:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} has no experience centroid")

ids_c = sorted(centroids.keys())
Xc = np.stack([centroids[rid] for rid in ids_c], axis=0)
target_c = centroids[TARGET_RESUME_ID].reshape(1, -1)
cent_sims = cosine_similarity(target_c, Xc)[0]

centroid_rank_df = (
    pd.DataFrame({"ResumeID": ids_c, "centroid_cosine_similarity": cent_sims.astype(float)})
    .sort_values("centroid_cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
meta_c = pd.read_csv(META_PATH)
centroid_rank_df = centroid_rank_df.merge(meta_c, on="ResumeID", how="left")

ti = ids_c.index(TARGET_RESUME_ID)
self_c = float(cent_sims[ti])
assert abs(self_c - 1.0) < 1e-5, f"expected target centroid self-similarity ~1, got {self_c}"
assert cent_sims.min() >= -1.0001 and cent_sims.max() <= 1.0001

centroid_rank_df

,ResumeID,centroid_cosine_similarity,primary_domain,role_family,seniority_band
0,1,1.000000,data_analytics,healthcare_analytics,mid
1,4,0.697838,data_analytics,healthcare_reporting,early
2,7,0.580856,data_science,research_ds,senior
3,9,0.579455,data_science,forecasting,mid
4,2,0.573389,data_analytics,bi_reporting,mid
5,5,0.555207,data_analytics,product_analytics,mid
6,14,0.544723,finance,fpna,senior
7,3,0.529107,data_analytics,operations_analytics,senior
8,6,0.525987,data_science,ml_engineering,mid
9,8,0.515832,data_science,nlp_search,mid


### LEFT OFF HERE Step 3 explain K


What “configurable k” means here
k is how many strongest per-bullet matches you keep when turning a full bullet–bullet matrix into one number per direction.

For each direction you first compute many scores, then you do not average all of them. You:

Take one score per bullet on that side (its best match on the other side).
Sort those scores high → low.
Keep only the top k (after capping—see below) and take their mean.
So k answers: “How much of the best overlap should count, before longer resumes dilute the signal?”

The plan makes k “configurable” by driving it from three knobs at the top of the cell (defaults in the plan):

BULLET_TOPK_MIN (e.g. 2) — never use fewer than this in the formula’s inner max (unless the resume has fewer bullets than that—then you use what you have).
BULLET_TOPK_FRACTION_OF_MIN (e.g. 0.4) — scale k with min(m, n), where m = target bullet count, n = other resume’s bullet count (for that pair).
BULLET_TOPK_MAX (e.g. 5) — cap k so it never gets huge on long resumes.
Formula (as in the plan):

k = min(max(BULLET_TOPK_MIN, floor(FRACTION * min(m, n))), BULLET_TOPK_MAX)

Then when taking the mean of “top k” scores:

T → R: you have m scores (one per target bullet). Use k_eff = min(k, m) scores.
R → T: you have n scores (one per R bullet). Use k_eff2 = min(k, n) scores.
“Configurable” means: you change BULLET_TOPK_MIN, BULLET_TOPK_FRACTION_OF_MIN, or BULLET_TOPK_MAX to make k larger or smaller for the same (m, n)—tuning how much you emphasize peak overlap vs breadth.

Your example: A has 10 bullets, B has 5
Fix which resume is the target (call it T). The other is R.

Case 1 — Target is A (10 bullets), other is B (5)
Then m = 10, n = 5, min(m, n) = 5.

With defaults MIN=2, FRACTION=0.4, MAX=5:

floor(0.4 * 5) = 2

max(2, 2) = 2

min(2, 5) = 2
So k = 2.

A → B: for each of 10 A-bulits, best cosine to some B-bullet → 10 numbers; keep the top 2 of those 10; mean of those 2.

B → A: for each of 5 B-bulits, best cosine to some A-bullet → 5 numbers; keep the top 2 of those 5; mean of those 2.

BulletMatchSim(A, B) = average of those two means.

Case 2 — Target is B (5 bullets), other is A (10)
Still min(10, 5) = 5, so k = 2 again. The matrix is 5×10 instead of 10×5, but min(m,n) is the same, so k is the same. The two directions still use m and n correctly for how many “per-bullet best” scores exist on each side.

Intuition
Smaller k (e.g. sticking near 2 for this pair): the score is driven mostly by the strongest alignments; weak bullets barely move it.
Larger k (raise FRACTION or MAX): you average more of the per-bullet bests, so the score reflects more of the resume’s bullets, not only the sharpest peaks.
So for 10 vs 5, with default knobs, k = 2 is the planned behavior for both directions’ top‑k means before they are combined.

### Step 3: Bullet-level alignment (concrete overlap)

**BulletMatchSim** compares **`TARGET_RESUME_ID`** to each other resume using **Experience** embeddings only. For a pair **(T, R)** with **m** and **n** bullets, build **M** = `cosine_similarity(X_T, X_R)` with shape **(m, n)**. **T→R:** per target row, take the **max** over **R**; keep the **top k** of those **m** values (mean). **R→T:** per **R** column, **max** over **T**; mean of top **k** of **n** values. **BulletMatchSim = (T→R + R→T) / 2**.

**Configurable k:** **`BULLET_TOPK_MIN`**, **`BULLET_TOPK_FRACTION_OF_MIN`**, **`BULLET_TOPK_MAX`** at the top of the next cell; **`k = min(max(MIN, floor(FRACTION * min(m,n))), MAX)`**, then **`k_eff = min(k, m)`** or **`min(k, n)`** per direction. Invalid **`MIN` / `MAX`** raises **`ValueError`**.

Without one-to-one matching, **one generic bullet** on one side can be the best match for **many** lines on the other; treat this as **component 3** of a future composite. Optional later: top bullet pairs for audit.

**Checks:** target **vs self** score **≈ 1**; scores in **[-1, 1]**.

![1BulletBestMatch](1BulletBestMatch.png)

![2BulletBestMatch](2BulletBestMatch.png)

![3BulletBestMatch](3BulletBestMatch.png)

In [6]:
import math

from sklearn.metrics.pairwise import cosine_similarity

# --- top-k size for bidirectional bullet aggregation ---
BULLET_TOPK_MIN = 2
BULLET_TOPK_MAX = 5
BULLET_TOPK_FRACTION_OF_MIN = 0.4


def bullet_topk_k(m: int, n: int, min_k: int, max_k: int, fraction: float) -> int:
    if min_k < 1:
        raise ValueError("BULLET_TOPK_MIN must be >= 1")
    if max_k < min_k:
        raise ValueError("BULLET_TOPK_MAX must be >= BULLET_TOPK_MIN")
    mn = min(m, n)
    return int(min(max(min_k, math.floor(fraction * mn)), max_k))


def mean_of_top_k_largest(scores_1d: np.ndarray, k: int) -> float:
    if scores_1d.size == 0:
        raise ValueError("empty bullet score vector")
    k_eff = min(k, int(scores_1d.size))
    desc = np.sort(scores_1d.astype(np.float64))[::-1][:k_eff]
    return float(np.mean(desc))


if not OUT_EMB_PATH.is_file():
    raise FileNotFoundError(f"Embedded corpus not found: {OUT_EMB_PATH.resolve()}")
if not META_PATH.is_file():
    raise FileNotFoundError(f"Resume metadata not found: {META_PATH.resolve()}")

emb_df3 = pd.read_csv(OUT_EMB_PATH, dtype={"ResumeID": int, "ItemID": int})
exp_mask3 = emb_df3["Section"].astype(str).str.strip() == "Experience"
exp_b = emb_df3.loc[exp_mask3 & emb_df3["embedding"].astype(str).str.len().gt(0)].copy()
exp_b = exp_b.sort_values(["ResumeID", "ItemID"], kind="mergesort")

all_ids_b = sorted(emb_df3["ResumeID"].unique())
exp_counts_b = exp_b.groupby("ResumeID").size()
missing_b = [rid for rid in all_ids_b if rid not in exp_counts_b.index or exp_counts_b.loc[rid] == 0]
if missing_b:
    raise ValueError(f"No non-empty Experience embedding for ResumeID(s): {missing_b}")

bullets_by_rid: dict[int, np.ndarray] = {}
for rid, grp in exp_b.groupby("ResumeID", sort=False):
    vecs = [np.array(json.loads(e), dtype=np.float32) for e in grp["embedding"]]
    bullets_by_rid[int(rid)] = np.stack(vecs, axis=0)

if TARGET_RESUME_ID not in bullets_by_rid:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} has no experience bullets")

ids_b = sorted(bullets_by_rid.keys())
X_T = bullets_by_rid[TARGET_RESUME_ID]

rows_bm = []
for rid in ids_b:
    X_R = bullets_by_rid[rid]
    m_tgt, n_other = X_T.shape[0], X_R.shape[0]
    k = bullet_topk_k(m_tgt, n_other, BULLET_TOPK_MIN, BULLET_TOPK_MAX, BULLET_TOPK_FRACTION_OF_MIN)
    M = cosine_similarity(X_T, X_R)
    t_to_r = mean_of_top_k_largest(np.max(M, axis=1), k)
    r_to_t = mean_of_top_k_largest(np.max(M, axis=0), k)
    bullet_sim = (t_to_r + r_to_t) / 2.0
    rows_bm.append({"ResumeID": rid, "bullet_match_similarity": bullet_sim})

self_bm = float(next(r["bullet_match_similarity"] for r in rows_bm if r["ResumeID"] == TARGET_RESUME_ID))

bullet_rank_df = (
    pd.DataFrame(rows_bm)
    .sort_values("bullet_match_similarity", ascending=False)
    .reset_index(drop=True)
)
meta_b = pd.read_csv(META_PATH)
bullet_rank_df = bullet_rank_df.merge(meta_b, on="ResumeID", how="left")

sims_bm = bullet_rank_df["bullet_match_similarity"].to_numpy(dtype=np.float64)
assert abs(self_bm - 1.0) < 1e-5, f"expected target bullet self-similarity ~1, got {self_bm}"
assert sims_bm.min() >= -1.0001 and sims_bm.max() <= 1.0001

bullet_rank_df

,ResumeID,bullet_match_similarity,primary_domain,role_family,seniority_band
0,1,1.000000,data_analytics,healthcare_analytics,mid
1,4,0.525314,data_analytics,healthcare_reporting,early
2,7,0.514245,data_science,research_ds,senior
3,2,0.461316,data_analytics,bi_reporting,mid
4,5,0.460672,data_analytics,product_analytics,mid
5,3,0.454597,data_analytics,operations_analytics,senior
6,6,0.441418,data_science,ml_engineering,mid
7,9,0.426676,data_science,forecasting,mid
8,13,0.394523,hr,total_rewards,mid
9,14,0.385076,finance,fpna,senior


### Step 4: Composite score (vs `TARGET_RESUME_ID`)

Merge the three signals from Steps 1–3 on **`ResumeID`**:

**FinalScore(T, R) = w₁·SummarySim + w₂·CentroidSim + w₃·BulletMatchSim** with **T = `TARGET_RESUME_ID`**.

Tune **`W_SUMMARY`**, **`W_CENTROID`**, **`W_BULLET`** at the top of the next cell (defaults **0.20 / 0.30 / 0.50**); they must **sum to 1** (or the cell raises). Output **`composite_rank_df`**: **`final_score`** plus the three component columns and metadata.

**Run Steps 1–3 first** so `rank_df`, `centroid_rank_df`, and `bullet_rank_df` exist.

### Step 4a: Top bullet pairs (evidence)

For the **top `EVIDENCE_TOP_RESUMES`** neighbors by **`final_score`** (excluding self), list the **`EVIDENCE_BULLET_PAIRS`** strongest **matrix cells** **(one target Experience row × one other Experience row)** by raw cosine similarity. So **`pair_rank_in_resume_pair`** 1…K means “K-th strongest *pair*,” **not** “other resume’s ItemID 1…K.” The table includes **full bullet text for the matching (other) resume** plus the target line for context. Same caveat as Step 3: one generic bullet on either side can dominate multiple pairs. Output **`bullet_evidence_df`**.

In [7]:
# --- composite weights (must sum to 1) ---
W_SUMMARY = 0.2
W_CENTROID = 0.3
W_BULLET = 0.5

_wsum = W_SUMMARY + W_CENTROID + W_BULLET
if abs(_wsum - 1.0) > 1e-6:
    raise ValueError(f"W_SUMMARY + W_CENTROID + W_BULLET must equal 1; got {_wsum}")

for _df, _col, _step in (
    (rank_df, "summary_cosine_similarity", "Step 1"),
    (centroid_rank_df, "centroid_cosine_similarity", "Step 2"),
    (bullet_rank_df, "bullet_match_similarity", "Step 3"),
):
    if _col not in _df.columns:
        raise RuntimeError(f"Missing {_col}; run {_step} cell first.")

composite_df = (
    rank_df[["ResumeID", "summary_cosine_similarity"]]
    .merge(centroid_rank_df[["ResumeID", "centroid_cosine_similarity"]], on="ResumeID")
    .merge(bullet_rank_df[["ResumeID", "bullet_match_similarity"]], on="ResumeID")
)
composite_df["final_score"] = (
    W_SUMMARY * composite_df["summary_cosine_similarity"]
    + W_CENTROID * composite_df["centroid_cosine_similarity"]
    + W_BULLET * composite_df["bullet_match_similarity"]
)

meta_f = pd.read_csv(META_PATH)
composite_rank_df = (
    composite_df.merge(meta_f, on="ResumeID", how="left")
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)

_self_final = float(
    composite_rank_df.loc[composite_rank_df["ResumeID"] == TARGET_RESUME_ID, "final_score"].iloc[0]
)
assert abs(_self_final - 1.0) < 1e-5, f"expected target final_score ~1, got {_self_final}"
_fs = composite_rank_df["final_score"].to_numpy(dtype=np.float64)
assert _fs.min() >= -1.0001 and _fs.max() <= 1.0001

composite_rank_df

,ResumeID,summary_cosine_similarity,centroid_cosine_similarity,bullet_match_similarity,final_score,primary_domain,role_family,seniority_band
0,1,1.000000,1.000000,1.000000,1.000000,data_analytics,healthcare_analytics,mid
1,4,0.515133,0.697838,0.525314,0.575035,data_analytics,healthcare_reporting,early
2,7,0.499845,0.580856,0.514245,0.531348,data_science,research_ds,senior
3,2,0.406697,0.573389,0.461316,0.484014,data_analytics,bi_reporting,mid
4,9,0.361967,0.579455,0.426676,0.459568,data_science,forecasting,mid
5,3,0.330743,0.529107,0.454597,0.452180,data_analytics,operations_analytics,senior
6,5,0.257279,0.555207,0.460672,0.448354,data_analytics,product_analytics,mid
7,8,0.444134,0.515832,0.363487,0.425320,data_science,nlp_search,mid
8,6,0.233892,0.525987,0.441418,0.425283,data_science,ml_engineering,mid
9,14,0.335894,0.544723,0.385076,0.423134,finance,fpna,senior


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

try:
    from IPython.display import display
except ImportError:
    display = print  # noqa: A001

# --- Step 4a: bullet text evidence for top neighbors by final_score ---
EVIDENCE_TOP_RESUMES = 5
EVIDENCE_BULLET_PAIRS = 5
# Short preview for the *target* bullet only; the matching resume uses full text in matching_resume_bullet_text.
TARGET_BULLET_SNIPPET_MAX_CHARS = 200

if "composite_rank_df" not in dir():
    raise RuntimeError("Run the Step 4 composite cell first (composite_rank_df).")

if not OUT_EMB_PATH.is_file():
    raise FileNotFoundError(f"Embedded corpus not found: {OUT_EMB_PATH.resolve()}")

emb_ev = pd.read_csv(OUT_EMB_PATH, dtype={"ResumeID": int, "ItemID": int})
emb_ev["Title"] = emb_ev["Title"].fillna("").astype(str)
exp_mask_ev = emb_ev["Section"].astype(str).str.strip() == "Experience"
exp_ev = emb_ev.loc[exp_mask_ev & emb_ev["embedding"].astype(str).str.len().gt(0)].copy()
exp_ev = exp_ev.sort_values(["ResumeID", "ItemID"], kind="mergesort")


def _full_bullet_label(title: str, text: str) -> str:
    title = (title or "").strip()
    text = (text or "").strip()
    return f"{title}: {text}" if title else text


def _maybe_shorten(label: str, max_chars: int) -> str:
    if len(label) <= max_chars:
        return label
    return label[: max_chars - 1] + "..."


records_ev: dict[int, list[dict]] = {}
for rid, grp in exp_ev.groupby("ResumeID", sort=False):
    recs = []
    for _, row in grp.iterrows():
        full_lbl = _full_bullet_label(row["Title"], row["Text"])
        recs.append(
            {
                "ItemID": int(row["ItemID"]),
                "bullet_text_full": full_lbl,
                "bullet_label_short": _maybe_shorten(full_lbl, TARGET_BULLET_SNIPPET_MAX_CHARS),
                "vec": np.array(json.loads(row["embedding"]), dtype=np.float32),
            }
        )
    records_ev[int(rid)] = recs

if TARGET_RESUME_ID not in records_ev:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} has no experience rows for evidence")

neighbor_ids_ev = (
    composite_rank_df.loc[composite_rank_df["ResumeID"] != TARGET_RESUME_ID]
    .head(EVIDENCE_TOP_RESUMES)["ResumeID"]
    .astype(int)
    .tolist()
)

T_recs = records_ev[TARGET_RESUME_ID]
X_T_ev = np.stack([r["vec"] for r in T_recs], axis=0)

evidence_rows_ev: list[dict] = []
for other_rid in neighbor_ids_ev:
    R_recs = records_ev[other_rid]
    X_R_ev = np.stack([r["vec"] for r in R_recs], axis=0)
    M_ev = cosine_similarity(X_T_ev, X_R_ev)
    flat = M_ev.ravel()
    top_lin = np.argsort(flat, kind="mergesort")[::-1][:EVIDENCE_BULLET_PAIRS]
    ii, jj = np.unravel_index(top_lin, M_ev.shape)
    for rank, (lin, i, j) in enumerate(zip(top_lin, ii, jj), start=1):
        ti, rj = T_recs[int(i)], R_recs[int(j)]
        evidence_rows_ev.append(
            {
                "other_ResumeID": other_rid,
                "pair_rank_in_resume_pair": rank,
                "cosine_sim": float(flat[lin]),
                "other_ItemID": rj["ItemID"],
                "matching_resume_bullet_text": rj["bullet_text_full"],
                "target_ItemID": ti["ItemID"],
                "target_bullet_preview": ti["bullet_label_short"],
            }
        )

bullet_evidence_df = pd.DataFrame(evidence_rows_ev)
_prev_mcw = pd.get_option("display.max_colwidth")
pd.set_option("display.max_colwidth", None)
try:
    display(bullet_evidence_df)
finally:
    pd.set_option("display.max_colwidth", _prev_mcw)


,other_ResumeID,pair_rank_in_resume_pair,cosine_sim,other_ItemID,matching_resume_bullet_text,target_ItemID,target_bullet_preview
0,4,1,0.534068,3,Analytics Intern: Visualized readmission trends in Tableau; supported monthly quality committee packets.,6,"Analyst: Data-Driven Insights and Quality Improvements: Analyzed and resolved complex bugs, conducted deep dives into data quality issues, and implemented solutions that improved reporting accuracy. ..."
1,4,2,0.516560,2,Healthcare Data Analyst: Pulled HEDIS-aligned cohorts in SQL; assisted clinical leaders with numerator/denominator logic and audit-ready documentation.,16,"Data Scientist: Provided data & dashboard to influence decision to use median procedure time vs ""one size fits all"" for high volume\nprocedures in cardiac surgery department."
2,4,3,0.498261,2,Healthcare Data Analyst: Pulled HEDIS-aligned cohorts in SQL; assisted clinical leaders with numerator/denominator logic and audit-ready documentation.,15,"Data Scientist: Architected, programmed & delivered seminal work-flow to calculate utilization of anesthesiologists site wide."
3,4,4,0.458536,2,Healthcare Data Analyst: Pulled HEDIS-aligned cohorts in SQL; assisted clinical leaders with numerator/denominator logic and audit-ready documentation.,13,"Analyst Intern: Automated calculation of Sensitivity, Specificity, Accuracy and Confidence Intervals for clinical study results."
4,4,5,0.440263,2,Healthcare Data Analyst: Pulled HEDIS-aligned cohorts in SQL; assisted clinical leaders with numerator/denominator logic and audit-ready documentation.,17,"Data Scientist: Created views, stored procedures and tables underpinning nurse scheduling optimization project; architected process\nneeded to ETL historical data and do daily refreshes of the data."
5,7,1,0.517807,3,Data Scientist: Built patient journey models from claims and EHR extracts; documented assumptions for regulatory discussions.,15,"Data Scientist: Architected, programmed & delivered seminal work-flow to calculate utilization of anesthesiologists site wide."
6,7,2,0.510683,2,Senior Data Scientist: Supported adaptive trial readouts; implemented reproducible R workflows for interim analyses under biostat review.,13,"Analyst Intern: Automated calculation of Sensitivity, Specificity, Accuracy and Confidence Intervals for clinical study results."
7,7,3,0.446694,3,Data Scientist: Built patient journey models from claims and EHR extracts; documented assumptions for regulatory discussions.,17,"Data Scientist: Created views, stored procedures and tables underpinning nurse scheduling optimization project; architected process\nneeded to ETL historical data and do daily refreshes of the data."
8,7,4,0.414888,3,Data Scientist: Built patient journey models from claims and EHR extracts; documented assumptions for regulatory discussions.,12,Analyst Intern: Evaluated certain patient physical characteristics and impact on prediction model accuracy.
9,7,5,0.393177,2,Senior Data Scientist: Supported adaptive trial readouts; implemented reproducible R workflows for interim analyses under biostat review.,12,Analyst Intern: Evaluated certain patient physical characteristics and impact on prediction model accuracy.
